# AuditKIT agentic integrations: AgentTune, Cohere, `hf:`

This one notebook runs the whole `tests/integrations_suite`:

1. **Offline tests.** AgentTune import (every output shape, run folders, provenance, the `agent:` endpoint), Cohere tool-call parsing and the real Cohere chat templates, and the `hf:` backend's tool, BOS and image paths. They use real tokenizers and templates; no model weights are loaded.
2. **Live `hf:`.** Qwen3-1.7B with native and prompt-mode tools (~4 GB, any GPU).
3. **Live Cohere.** Tiny Aya Global in prompt mode, plus the refusal of native mode (~7 GB). **Gated**, so you need to accept the licence on the Hub.
4. **Live Command R7B** with native tools (~16 GB). **Gated**, and it needs a 24 GB+ GPU (L4 or A100).

**Before running:**
- Runtime → Change runtime type → **GPU**.
- Add these Colab secrets (🔑 in the left bar):
  - `GITHUB_TOKEN`, with read access to `aryapratinavseth/AuditKIT-Internal`;
  - `HF_TOKEN`, from an account that has accepted the licences for `CohereLabs/tiny-aya-global` and `CohereLabs/c4ai-command-r7b-12-2024`.

Expected result: the offline tests pass except the known findings, which show as `xfail` (listed in `tests/integrations_suite/README.md`). The live cells print each model's outputs and scores.

In [ ]:
# Settings
BRANCH = "test/agentic-rag-full-suite"   # the PR branch with tests/integrations_suite (stacked on HK-AK/interop)
RUN_COHERE = True                    # needs HF_TOKEN + accepted licence
RUN_R7B = "auto"                     # "auto" = only when the GPU has >= 24 GB

In [ ]:
import os, subprocess
from google.colab import userdata
gh = userdata.get("GITHUB_TOKEN")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN secret not set: the Cohere cells will be skipped")
if not os.path.isdir("AuditKIT-Internal"):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH,
                    f"https://{gh}@github.com/aryapratinavseth/AuditKIT-Internal.git"], check=True)
%cd AuditKIT-Internal
!git log --oneline -1
!pip -q install -e ".[transformers]" pytest
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# The offline tests load real tokenizers from the local HF cache (local_files_only),
# so fetch just the tokenizer files once (no weights).
from huggingface_hub import snapshot_download
for repo in ["gpt2", "Qwen/Qwen3-1.7B", "NousResearch/Llama-2-7b-hf"]:
    snapshot_download(repo, allow_patterns=["*.json", "*.txt", "*.model", "tokenizer*"])
print("tokenizers cached")

## 1 · Offline integration tests (AgentTune, Cohere, `hf:`)

In [ ]:
!python -m pytest -o addopts= tests/integrations_suite -q -rx -p no:cacheprovider --ignore=tests/integrations_suite/test_live.py

## 2 · Live `hf:`: Qwen3-1.7B with native and prompt-mode tools

In [ ]:
!AK_LIVE_HF=1 python -m pytest -o addopts= tests/integrations_suite/test_live.py -k qwen3 -q -s -W ignore -p no:cacheprovider

## 3 · Live Cohere: Tiny Aya Global (gated)

In [ ]:
if RUN_COHERE and os.environ.get("HF_TOKEN"):
    !AK_LIVE_COHERE=1 python -m pytest -o addopts= tests/integrations_suite/test_live.py -k tiny_aya -q -s -W ignore -p no:cacheprovider
else:
    print("skipped: set HF_TOKEN (and accept the licence) to run")

## 4 · Live Cohere: Command R7B with native tools (gated, 24 GB+ GPU)

In [ ]:
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
go = RUN_R7B is True or (RUN_R7B == "auto" and mem >= 23)
if go and os.environ.get("HF_TOKEN"):
    !AK_LIVE_R7B=1 python -m pytest -o addopts= tests/integrations_suite/test_live.py -k r7b -q -s -W ignore -p no:cacheprovider
else:
    print(f"skipped: GPU memory {mem:.0f} GB (needs >= 24) or no HF_TOKEN")

## Reading the results

- **Offline:** `passed` plus `xfailed`. Every `xfail` is a known finding with its reason, and the README lists them. An `XPASS` means a finding was fixed, so remove its marker.
- **Live:** each line shows the sample id and the tail of the raw model output. The headline shows `tool_call_f1`, validity and parallel scores. For Command R7B, the raw output is the special-token-stripped action list, which the Cohere parser reads.